In [ ]:
# Install dependencies
!pip install datasets tiktoken torch

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken
import os

In [ ]:
# hyperparameters (Must precisely match the trained model)
batch_size = 1 # Inference
block_size = 256
n_embd = 384
n_head = 6
n_layer = 6
head_dim = n_embd // n_head
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab + 1 # +1 for mask token
mask_token_id = enc.n_vocab

def encode(s):
    return enc.encode(s)

def decode(l):
    return enc.decode([t for t in l if t != mask_token_id])

In [ ]:
# Base Model Architecture Definitions
def norm(x):
    return F.rms_norm(x, (x.size(-1),))

def apply_rotary_emb(x, cos, sin):
    assert x.ndim == 4
    d = x.shape[3] // 2
    x1, x2 = x[..., :d], x[..., d:]
    y1 = x1 * cos + x2 * sin
    y2 = x1 * (-sin) + x2 * cos
    return torch.cat([y1, y2], 3).to(x.dtype)

class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_q = nn.Linear(n_embd, n_embd, bias=False)
        self.c_k = nn.Linear(n_embd, n_embd, bias=False)
        self.c_v = nn.Linear(n_embd, n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x, cos_sin):
        B, T, C = x.size()
        q = self.c_q(x).view(B, T, n_head, head_dim)
        k = self.c_k(x).view(B, T, n_head, head_dim)
        v = self.c_v(x).view(B, T, n_head, head_dim)
        cos, sin = cos_sin
        q, k = apply_rotary_emb(q, cos, sin), apply_rotary_emb(k, cos, sin)
        q, k = norm(q), norm(k)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        y = y.transpose(1, 2).contiguous().view(B, T, -1)
        return self.c_proj(y)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        hidden_dim = int(8 * n_embd / 3) 
        self.w1 = nn.Linear(n_embd, hidden_dim, bias=False)
        self.w2 = nn.Linear(n_embd, hidden_dim, bias=False)
        self.c_proj = nn.Linear(hidden_dim, n_embd, bias=False)

    def forward(self, x):
        return self.c_proj(F.silu(self.w1(x)) * self.w2(x))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = MultiHeadAttention()
        self.mlp = MLP()

    def forward(self, x, cos_sin):
        x = x + self.attn(norm(x), cos_sin)
        x = x + self.mlp(norm(x))
        return x

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.rotary_seq_len = block_size * 2
        
        device_for_emb = torch.device(device)
        channel_range = torch.arange(0, head_dim, 2, dtype=torch.float32, device=device_for_emb)
        inv_freq = 1.0 / (10000 ** (channel_range / head_dim))
        t = torch.arange(self.rotary_seq_len, dtype=torch.float32, device=device_for_emb)
        freqs = torch.outer(t, inv_freq)
        cos, sin = freqs.cos(), freqs.sin()
        cos, sin = cos[None, :, None, :], sin[None, :, None, :]
        
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.blocks = nn.ModuleList([Block() for _ in range(n_layer)])
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.size()
        x = norm(self.token_emb(idx))
        cos_sin = (self.cos[:, :T], self.sin[:, :T])
        for block in self.blocks:
            x = block(x, cos_sin)
        return self.lm_head(norm(x)), None

In [ ]:
@torch.no_grad()
def generate(model, prompt_tokens, max_new_tokens, temp=1.0, confidence_threshold=0.95, top_k=3):
    model.eval()
    prompt_len = len(prompt_tokens)
    all_tokens = prompt_tokens.copy()
    total_steps = 0

    while len(all_tokens) - prompt_len < max_new_tokens:
        block_len = min(block_size - prompt_len, prompt_len + max_new_tokens - len(all_tokens))
        
        x = torch.full((1, block_size), mask_token_id, dtype=torch.long, device=device)
        x[0, :prompt_len] = torch.tensor(all_tokens[-prompt_len:], device=device)
        
        masked = torch.zeros(1, block_size, dtype=torch.bool, device=device)
        masked[0, prompt_len:prompt_len + block_len] = True

        while masked.any():
            total_steps += 1
            logits, _ = model(x)
            probs = F.softmax(logits / temp, dim=-1)
            top_k_probs, top_k_indices = torch.topk(probs, k=top_k, dim=-1)
            confidences = top_k_probs.sum(dim=-1)

            decode_mask = (confidences >= confidence_threshold) & masked
            if not decode_mask.any():
                masked_confidences = torch.where(masked, confidences, torch.tensor(-float('inf')).to(device))
                decode_mask.view(-1)[masked_confidences.argmax()] = True

            top_k_probs_norm = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
            sampled_k = torch.multinomial(top_k_probs_norm.view(-1, top_k), 1).view(1, block_size)
            sampled_tokens = torch.gather(top_k_indices, -1, sampled_k.unsqueeze(-1)).squeeze(-1)

            x = torch.where(decode_mask, sampled_tokens, x)
            masked = masked & ~decode_mask
            
        all_tokens.extend(x[0, prompt_len:prompt_len + block_len].tolist())
        prompt_len = len(all_tokens)
        
    return decode(all_tokens)

In [ ]:
# Initialize model and load weights
weights_path = '/model/tinystories_diffusion.pt'
model = Model().to(device)

if os.path.exists(weights_path):
    print(f"Loading weights from {weights_path}...")
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print("Model loaded successfully!")
else:
    print(f"ERROR: {weights_path} not found. Please make sure you have fully run the training notebook first!")

In [ ]:
# Run Inference Demonstration
def run_interactive_inference():
    print("=== TinyStories Diffusion Inference ===")
    print("Type your prompt (~10 words), or 'quit' to exit.")
    
    while True:
        user_input = input("\nPrompt: ")
        if user_input.lower() in ['quit', 'exit', 'q']:
            break
            
        prompt_tokens = encode(user_input)
        print(f"\n[Generating ~100 tokens from '{user_input}'...]")
        
        # Adjust max_new_tokens or temp as needed
        output = generate(model, prompt_tokens, max_new_tokens=100, temp=1.0)
        
        print("\n--- Output ---")
        print(output)
        print("--------------\n")

# Run the interactive loop
run_interactive_inference()